# Power Dominating Set via Spanning Tree Heuristic
**Approach:**
1. Build a DFS spanning tree from the general graph
2. Solve PDS on that tree (bottom-up greedy state machine)
3. Verify the resulting set is a valid PDS on the original graph

**Why it works:** Any PDS that observes the spanning tree also observes the original graph — non-tree edges only *help* propagation (they never block observation).

In [1]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import time
import pandas as pd
from scipy.io import mmread
from collections import defaultdict, deque

print("All libraries loaded successfully.")

All libraries loaded successfully.


## Step 1: Build DFS Spanning Tree

In [2]:
def build_dfs_spanning_tree(G, root=None):
    """
    Build a DFS spanning tree of G rooted at `root`.
    Returns (root, parent, children).
    """
    if root is None:
        # Pick the highest-degree vertex for a better-shaped tree
        root = max(G.nodes(), key=lambda v: G.degree(v))

    parent   = {root: None}
    children = defaultdict(list)
    visited  = {root}
    stack    = [root]

    while stack:
        v = stack.pop()
        for u in G.neighbors(v):
            if u not in visited:
                visited.add(u)
                parent[u] = v
                children[v].append(u)
                stack.append(u)

    return root, parent, children

## Step 2: Simulate PDS Propagation on a Graph
Given a monitor set S, apply PDS observation rules and return the set of all observed vertices.

In [3]:
def simulate_propagation(G, S):
    """
    Apply PDS rules on G starting from monitor set S:
      Rule 1 (Domination):   every v in S observes itself and all its neighbors.
      Rule 2 (Propagation):  if an observed v has exactly 1 unobserved neighbor u,
                             then u becomes observed. Repeat until stable.
    Returns the set of observed vertices.
    """
    observed = set()

    # Rule 1
    for v in S:
        observed.add(v)
        observed.update(G.neighbors(v))

    # Rule 2 - fixed-point iteration
    changed = True
    while changed:
        changed = False
        for v in list(observed):
            unobs = [u for u in G.neighbors(v) if u not in observed]
            if len(unobs) == 1:
                observed.add(unobs[0])
                changed = True

    return observed

## Step 3: Solve PDS on the Spanning Tree (Bottom-Up Greedy)

Each vertex gets one of three **states** after its subtree is processed:

| State          | Meaning                                                                                       |
|----------------|-----------------------------------------------------------------------------------------------|
| `in_S`         | The vertex is in S; it and all its neighbors are observed.                                    |
| `propagating`  | Exactly one downward child still needs help; propagation through v will resolve it.           |
| `unobserved`   | v has no help from below; relies on its parent to be observed.                                |

**Greedy rule:** at each internal vertex v, count children whose state is *not* `in_S` (call them *needy*).
- More than 1 needy child -> put **v** in S (Rule 2 can push to only one neighbor, so we need Rule 1 here).
- Exactly 1 needy child -> v propagates down to it; v becomes `propagating`.
- 0 needy children -> all children are `in_S`, so v is already observed as their neighbor.

Root handling: if root isn't `in_S` and has no `in_S` child, add root to S.

In [4]:
def pds_on_tree(root, children):
    """
    Bottom-up greedy PDS on a rooted tree.
    Returns the PDS set S.
    """
    # Build BFS order from root, then reverse -> post-order (leaves first)
    order = []
    queue = deque([root])
    while queue:
        v = queue.popleft()
        order.append(v)
        queue.extend(children[v])
    order.reverse()

    S = set()
    state = {}   # 'in_S' | 'propagating' | 'unobserved'

    for v in order:
        kids = children[v]

        if not kids:
            # Leaf - relies on parent for observation
            state[v] = 'unobserved'
            continue

        # 'needy' kids: those that still need v's help (i.e., not in_S themselves)
        needy = [c for c in kids if state[c] != 'in_S']

        if len(needy) >= 2:
            # Multiple needy children - v must go into S to dominate all of them
            S.add(v)
            state[v] = 'in_S'
        elif len(needy) == 1:
            # Single needy child - propagation from v handles it
            state[v] = 'propagating'
        else:
            # All kids are in S - v is already observed (neighbor of an S-vertex)
            state[v] = 'unobserved'

    # Root handling: if root is not in S and has no in_S child, add it
    if state[root] != 'in_S':
        if not any(state[c] == 'in_S' for c in children[root]):
            S.add(root)

    return S

## Step 4: Main Heuristic - PDS on General Graph via Spanning Tree

Wraps the three pieces together:
1. Build the DFS spanning tree T of G.
2. Solve PDS on T greedily.
3. Verify the result on the original graph G.

Returns `(S, observed_G, is_valid_G)`. The optional `verbose` flag controls per-call printing - disable it during batch experiments.

In [5]:
def pds_spanning_tree_heuristic(G, root=None, verbose=True):
    # Step 1
    root, parent, children = build_dfs_spanning_tree(G, root)

    G_tree = nx.Graph()
    G_tree.add_nodes_from(G.nodes())
    for v, p in parent.items():
        if p is not None:
            G_tree.add_edge(v, p)

    # Step 2
    S = pds_on_tree(root, children)

    # Step 3 - verify on original G
    observed_G    = simulate_propagation(G, S)
    observed_tree = simulate_propagation(G_tree, S)
    is_valid_G    = (observed_G == set(G.nodes()))
    is_valid_tree = (observed_tree == set(G_tree.nodes()))

    if verbose:
        print(f"Graph       : {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
        print(f"Spanning T  : {G_tree.number_of_nodes()} nodes, {G_tree.number_of_edges()} edges  (root = {root})")
        print(f"PDS S       : {sorted(S)}   (size = {len(S)})")
        print(f"Valid on T  : {is_valid_tree}")
        print(f"Valid on G  : {is_valid_G}")
        if not is_valid_G:
            print(f"  !! Missed on G: {sorted(set(G.nodes()) - observed_G)}")

    return S, observed_G, is_valid_G

## Step 5: Load and Inspect a Sample Dataset

In [6]:
def load_graph(filepath):
    """
    Load a graph from a Matrix Market (.mtx or .mtx.gz) file.
    Removes self-loops since they are irrelevant to PDS.
    """
    A = mmread(filepath)
    G = nx.from_scipy_sparse_array(A)
    G.remove_edges_from(nx.selfloop_edges(G))
    return G


# Quick inspection of one dataset
G = load_graph("datasets/bcspwr01.mtx.gz")

print(f"Nodes      : {G.number_of_nodes()}")
print(f"Edges      : {G.number_of_edges()}")
print(f"Avg degree : {sum(dict(G.degree()).values()) / G.number_of_nodes():.2f}")
print(f"Max degree : {max(G.degree(), key=lambda x: x[1])}")

Nodes      : 39
Edges      : 46
Avg degree : 2.36
Max degree : (15, 5)


## Step 6: Batch Run on BCSPWR Dataset Family

In [10]:
datasets = [
    ("BCSPWR01", "datasets/bcspwr01.mtx.gz"),
    ("BCSPWR02", "datasets/bcspwr02.mtx.gz"),
    ("BCSPWR03", "datasets/bcspwr03.mtx.gz"),
    ("BCSPWR04", "datasets/bcspwr04.mtx.gz"),
    ("BCSPWR05", "datasets/bcspwr05.mtx.gz"),
    ("BCSPWR06", "datasets/bcspwr06.mtx.gz"),
    ("BCSPWR07", "datasets/bcspwr07.mtx.gz"),
    ("BCSPWR08", "datasets/bcspwr08.mtx.gz"),
    ("BCSPWR09", "datasets/bcspwr09.mtx.gz"),
    ("BCSPWR10", "datasets/bcspwr10.mtx.gz"),
]

results = []

In [11]:
for name, path in datasets:
    try:
        G_i = load_graph(path)

        # Spanning Tree Heuristic
        t0 = time.time()
        pds_tree, _, valid_tree = pds_spanning_tree_heuristic(G_i, verbose=False)
        t_tree = time.time() - t0

        results.append({
            "Dataset"      : name,
            "Nodes"        : G_i.number_of_nodes(),
            "Edges"        : G_i.number_of_edges(),
            "SpanTree PDS" : len(pds_tree),
            "PDS Ratio"    : round(len(pds_tree) / G_i.number_of_nodes(), 4),
            "Valid"        : valid_tree,
            "Time(s)"      : round(t_tree, 4),
        })

        print(f"{name}: SpanTree_PDS={len(pds_tree)} ({t_tree:.4f}s, valid={valid_tree})")

    except FileNotFoundError:
        print(f"{name}: file not found - skipping")

BCSPWR01: SpanTree_PDS=8 (0.0003s, valid=True)
BCSPWR02: SpanTree_PDS=10 (0.0004s, valid=True)
BCSPWR03: SpanTree_PDS=26 (0.0008s, valid=True)
BCSPWR04: SpanTree_PDS=49 (0.0027s, valid=True)
BCSPWR05: SpanTree_PDS=91 (0.0038s, valid=True)
BCSPWR06: SpanTree_PDS=269 (0.0163s, valid=True)
BCSPWR07: SpanTree_PDS=308 (0.0175s, valid=True)
BCSPWR08: SpanTree_PDS=301 (0.0820s, valid=True)
BCSPWR09: SpanTree_PDS=325 (0.0203s, valid=True)
BCSPWR10: SpanTree_PDS=1054 (0.0811s, valid=True)


### Final Results Table

In [12]:
df = pd.DataFrame(results)
df

,Dataset,Nodes,Edges,SpanTree PDS,PDS Ratio,Valid,Time(s)
0,BCSPWR01,39,46,8,0.2051,True,0.0003
1,BCSPWR02,49,59,10,0.2041,True,0.0004
2,BCSPWR03,118,179,26,0.2203,True,0.0008
3,BCSPWR04,274,669,49,0.1788,True,0.0027
4,BCSPWR05,443,590,91,0.2054,True,0.0038
5,BCSPWR06,1454,1923,269,0.1850,True,0.0163
6,BCSPWR07,1612,2106,308,0.1911,True,0.0175
7,BCSPWR08,1624,2213,301,0.1853,True,0.0820
8,BCSPWR09,1723,2394,325,0.1886,True,0.0203
9,BCSPWR10,5300,8271,1054,0.1989,True,0.0811
